In [70]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split


In [71]:
def load_data(file_name):
    df = pd.read_csv(file_name)
    df.dropna(inplace=True)
    return df

bitcoin_df = load_data("bitcoin_processed.csv")
ethereum_df = load_data("ethereum_processed.csv")


In [72]:
scaler = MinMaxScaler()

def preprocess_data(df):
    features = ['open', 'high', 'low', 'volume', 'quote_asset_volume',
                'number_of_trades', 'taker_buy_base_asset_volume',
                'taker_buy_quote_asset_volume', 'average_price', 'price_change']
    target = 'target_close'
    
    df[features] = scaler.fit_transform(df[features])
    df[target] = scaler.fit_transform(df[[target]])
    
    return df, features, target

bitcoin_df, features, target = preprocess_data(bitcoin_df)
ethereum_df, _, _ = preprocess_data(ethereum_df)


In [73]:
def split_data(df, features, target):
    X = df[features]
    y = df[target]
    return train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)

X_btc_train, X_btc_test, y_btc_train, y_btc_test = split_data(bitcoin_df, features, target)
X_eth_train, X_eth_test, y_eth_train, y_eth_test = split_data(ethereum_df, features, target)


In [74]:
def train_lgbm(X_train, y_train):
    lgb_train = lgb.Dataset(X_train, label=y_train)
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'learning_rate': 0.05,
        'num_leaves': 31,
        'max_depth': -1,
        'verbose': -1
    }
    model = lgb.train(params, lgb_train, num_boost_round=100)
    return model


In [75]:
btc_model = train_lgbm(X_btc_train, y_btc_train)
eth_model = train_lgbm(X_eth_train, y_eth_train)


In [76]:
def evaluate_model(model, X_test, y_test):
    predictions = model.predict(X_test)
    mse = mean_squared_error(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mse)
    return mse, mae, rmse


In [77]:
mse_btc, mae_btc, rmse_btc = evaluate_model(btc_model, X_btc_test, y_btc_test)
print(f'Bitcoin - MSE: {mse_btc:.6f}, MAE: {mae_btc:.6f}, RMSE: {rmse_btc:.6f}')

mse_eth, mae_eth, rmse_eth = evaluate_model(eth_model, X_eth_test, y_eth_test)
print(f'Ethereum - MSE: {mse_eth:.6f}, MAE: {mae_eth:.6f}, RMSE: {rmse_eth:.6f}')


Bitcoin - MSE: 0.052853, MAE: 0.172704, RMSE: 0.229898
Ethereum - MSE: 0.001706, MAE: 0.031865, RMSE: 0.041301


In [78]:
def evaluate_model_with_inverse_scaling(model, X_test, y_test, scaler):
    predictions = model.predict(X_test)
    
    y_test_actual = scaler.inverse_transform(y_test.values.reshape(-1, 1))
    predictions_actual = scaler.inverse_transform(predictions.reshape(-1, 1))
    
    mse = mean_squared_error(y_test_actual, predictions_actual)
    mae = mean_absolute_error(y_test_actual, predictions_actual)
    rmse = np.sqrt(mse)
    
    return mse, mae, rmse, y_test_actual, predictions_actual


In [79]:
mse_btc_actual, mae_btc_actual, rmse_btc_actual, y_true_btc, y_pred_btc = evaluate_model_with_inverse_scaling(btc_model, X_btc_test, y_btc_test, scaler)
print(f'Bitcoin (Actual Values) - MSE: {mse_btc_actual:.6f}, MAE: {mae_btc_actual:.6f}, RMSE: {rmse_btc_actual:.6f}')

mse_eth_actual, mae_eth_actual, rmse_eth_actual, y_true_eth, y_pred_eth = evaluate_model_with_inverse_scaling(eth_model, X_eth_test, y_eth_test, scaler)
print(f'Ethereum (Actual Values) - MSE: {mse_eth_actual:.6f}, MAE: {mae_eth_actual:.6f}, RMSE: {rmse_eth_actual:.6f}')


Bitcoin (Actual Values) - MSE: 463724.165962, MAE: 511.562430, RMSE: 680.972955
Ethereum (Actual Values) - MSE: 14966.537155, MAE: 94.386582, RMSE: 122.337799
